# Memory-Efficient Embedding Optimizer

- Auto list all CSV files from folder
- Show one sample file column names + dtypes
- Process all files sequentially (one file at a time)
- Save with same name in same folder (in-place, safe temp replace)
- Print per-file processing and storage improvement


In [2]:
import gc
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
    HAS_PYARROW = True
except Exception:
    HAS_PYARROW = False

pd.set_option('display.max_columns', 200)
print('Imports loaded. pyarrow available:', HAS_PYARROW)

Imports loaded. pyarrow available: True


In [3]:
def _parse_embedding_string(value: object) -> Tuple[np.ndarray, bool]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.empty(0, dtype=np.float32), False

    text = str(value).strip()
    if not text:
        return np.empty(0, dtype=np.float32), False

    if text.startswith('[') and text.endswith(']'):
        text = text[1:-1].strip()

    text = text.replace(';', ',').replace('\n', ' ')

    arr = np.fromstring(text, sep=',', dtype=np.float32)
    if arr.size == 0 and text:
        arr = np.fromstring(text, sep=' ', dtype=np.float32)

    ok = arr.size > 0
    return arr, ok


def list_csv_files(folder_path: str, pattern: str = '*.csv') -> list:
    folder = Path(folder_path)
    if not folder.exists():
        raise FileNotFoundError(f'Folder not found: {folder_path}')

    files = sorted(folder.glob(pattern))
    print(f'Found {len(files)} CSV files in: {folder}')
    for i, f in enumerate(files, start=1):
        print(f'  [{i}] {f.name}')
    return files


def inspect_file(file_path: str, sample_rows: int = 2) -> None:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f'File not found: {file_path}')

    sample_df = pd.read_csv(path, nrows=max(sample_rows, 2), low_memory=True)

    print('Sample file:', path.name)
    print('\nColumns:')
    print(list(sample_df.columns))
    print('\nDtypes (sample-based):')
    print(sample_df.dtypes)
    print(f'\nFirst {sample_rows} rows:')
    display(sample_df.head(sample_rows))


def convert_embedding_column(chunk: pd.DataFrame, embedding_col: str) -> Tuple[pd.DataFrame, Dict[str, int]]:
    if embedding_col not in chunk.columns:
        raise KeyError(f'Missing embedding column: {embedding_col}')

    parsed = chunk[embedding_col].map(_parse_embedding_string)
    arrays = parsed.map(lambda t: t[0])
    ok_flags = parsed.map(lambda t: t[1])

    chunk[embedding_col] = arrays

    stats = {
        'rows': int(len(chunk)),
        'parsed_ok': int(ok_flags.sum()),
        'parsed_failed': int((~ok_flags).sum()),
        'empty_vectors': int(arrays.map(lambda a: a.size == 0).sum())
    }
    return chunk, stats


def process_file(
    file_path: str,
    embedding_col: str,
    output_path: Optional[str] = None,
    chunksize: int = 10000,
    output_format: str = 'csv',
    overwrite: bool = True,
    in_place: bool = True
) -> Dict[str, object]:
    input_file = Path(file_path)
    if not input_file.exists():
        raise FileNotFoundError(f'File not found: {file_path}')

    if output_format not in {'parquet', 'csv'}:
        raise ValueError("output_format must be 'parquet' or 'csv'")

    if output_format == 'parquet' and not HAS_PYARROW:
        raise RuntimeError('pyarrow is required for parquet output')

    before_size = input_file.stat().st_size

    if in_place:
        if output_format == 'csv':
            output_file = input_file
            tmp_file = input_file.with_name(f'{input_file.stem}.__tmp__.csv')
        else:
            output_file = input_file.with_suffix('.parquet')
            tmp_file = output_file.with_name(f'{output_file.stem}.__tmp__.parquet')
    else:
        if output_path is None:
            suffix = '.parquet' if output_format == 'parquet' else '.csv'
            output_file = input_file.with_name(f'{input_file.stem}_optimized{suffix}')
        else:
            output_file = Path(output_path)
        tmp_file = output_file.with_name(f'{output_file.stem}.__tmp__{output_file.suffix}')

    if tmp_file.exists():
        tmp_file.unlink()

    if output_file.exists() and (not overwrite) and (not in_place):
        raise FileExistsError(f'Output exists: {output_file}. Set overwrite=True.')

    print(f'Processing file: {input_file.name}')

    total_rows = 0
    total_ok = 0
    total_failed = 0
    chunk_idx = 0

    parquet_writer = None

    for chunk in pd.read_csv(input_file, chunksize=chunksize, low_memory=True):
        chunk_idx += 1
        chunk, stats = convert_embedding_column(chunk, embedding_col)

        if output_format == 'parquet':
            chunk[embedding_col] = chunk[embedding_col].map(lambda a: a.tolist())
            table = pa.Table.from_pandas(chunk, preserve_index=False)
            if parquet_writer is None:
                parquet_writer = pq.ParquetWriter(tmp_file, table.schema)
            parquet_writer.write_table(table)
        else:
            # Save arrays as compact string in csv, same file name rewrite.
            chunk[embedding_col] = chunk[embedding_col].map(
                lambda a: '[' + ','.join(np.char.mod('%.8g', a)) + ']'
            )
            mode = 'w' if chunk_idx == 1 else 'a'
            header = chunk_idx == 1
            chunk.to_csv(tmp_file, index=False, mode=mode, header=header)

        total_rows += stats['rows']
        total_ok += stats['parsed_ok']
        total_failed += stats['parsed_failed']

        del chunk
        gc.collect()

    if parquet_writer is not None:
        parquet_writer.close()

    if in_place and output_format == 'csv':
        tmp_file.replace(input_file)
        final_output = input_file
    else:
        if output_file.exists() and overwrite:
            output_file.unlink()
        tmp_file.replace(output_file)
        final_output = output_file

    after_size = final_output.stat().st_size
    saved_bytes = before_size - after_size
    saved_pct = (saved_bytes / before_size * 100.0) if before_size > 0 else 0.0

    result = {
        'file': str(input_file),
        'output': str(final_output),
        'rows': total_rows,
        'parsed_ok': total_ok,
        'parsed_failed': total_failed,
        'size_before_bytes': int(before_size),
        'size_after_bytes': int(after_size),
        'saved_bytes': int(saved_bytes),
        'saved_pct': float(saved_pct),
        'status': 'success'
    }

    print(f"Done: {input_file.name} | Saved: {saved_bytes} bytes ({saved_pct:.2f}%)")
    return result


def process_folder(
    folder_path: str,
    embedding_col: str,
    pattern: str = '*.csv',
    chunksize: int = 10000,
    output_format: str = 'csv',
    overwrite: bool = True,
    in_place: bool = True
) -> pd.DataFrame:
    files = list_csv_files(folder_path=folder_path, pattern=pattern)
    if not files:
        return pd.DataFrame()

    results = []

    for idx, file_path in enumerate(files, start=1):
        print(f'\n[{idx}/{len(files)}] Processing: {file_path.name}')
        try:
            result = process_file(
                file_path=str(file_path),
                embedding_col=embedding_col,
                output_path=None,
                chunksize=chunksize,
                output_format=output_format,
                overwrite=overwrite,
                in_place=in_place
            )
        except Exception as exc:
            result = {
                'file': str(file_path),
                'output': None,
                'rows': 0,
                'parsed_ok': 0,
                'parsed_failed': 0,
                'size_before_bytes': int(file_path.stat().st_size) if file_path.exists() else 0,
                'size_after_bytes': 0,
                'saved_bytes': 0,
                'saved_pct': 0.0,
                'status': 'failed',
                'error': str(exc)
            }
            print('Failed:', exc)

        results.append(result)
        gc.collect()

    summary = pd.DataFrame(results)

    total_before = int(summary['size_before_bytes'].sum()) if 'size_before_bytes' in summary else 0
    total_after = int(summary['size_after_bytes'].sum()) if 'size_after_bytes' in summary else 0
    total_saved = total_before - total_after
    total_saved_pct = (total_saved / total_before * 100.0) if total_before > 0 else 0.0

    print('\n=== Storage Improvement Summary ===')
    print(f'Total before: {total_before} bytes')
    print(f'Total after : {total_after} bytes')
    print(f'Total saved : {total_saved} bytes ({total_saved_pct:.2f}%)')

    display(summary)
    return summary


print('Functions ready: list_csv_files, inspect_file, convert_embedding_column, process_file, process_folder')

Functions ready: list_csv_files, inspect_file, convert_embedding_column, process_file, process_folder


## Step 1: Load all CSV names and inspect one sample file

In [7]:
folder_path = '/home/hp/SEM2/INLP/Naretve_Shift/Processed_Data/Topic_Wise_w5'
csv_files = list_csv_files(folder_path, pattern='*.csv')
if not csv_files:
    raise ValueError(f'No CSV files found in: {folder_path}')

sample_file_path = str(csv_files[0])
print('\nSample file selected:', Path(sample_file_path).name)
inspect_file(sample_file_path, sample_rows=2)

Found 5 CSV files in: /home/hp/SEM2/INLP/Naretve_Shift/Processed_Data/Topic_Wise_w5
  [1] Climate.csv
  [2] Economics.csv
  [3] Health.csv
  [4] Technology.csv
  [5] War.csv

Sample file selected: Climate.csv
Sample file: Climate.csv

Columns:
['date', 'sentence_id', 'main_sentence', 'w5_embedding', 'War', 'Health', 'Technology', 'Climate', 'Economics']

Dtypes (sample-based):
date                 str
sentence_id          str
main_sentence        str
w5_embedding         str
War              float64
Health           float64
Technology       float64
Climate          float64
Economics        float64
dtype: object

First 2 rows:


,date,sentence_id,main_sentence,w5_embedding,War,Health,Technology,Climate,Economics
0,2011-09-20 19:55:07,f1_a47_s2,"""The seven, members of a so-called ""major risk...","-0.016366452,0.12162542,-0.006107007,0.0451595...",0.249266,0.274897,0.187701,0.301356,0.200209
1,2011-09-21 13:59:11,f1_a50_s1,Story highlightsUrsula Sladek created Germany'...,"-0.02180171,0.017887823,0.010610466,0.04336527...",0.180741,0.126097,0.217226,0.444158,0.151267


## Step 2: User input embedding column

In [8]:
embedding_column = input('Which column is the embedding column? ').strip()
print('Embedding column selected:', embedding_column)

Embedding column selected: w5_embedding


## Step 3: Process all files in folder (same names, same folder)

In [9]:
folder_summary = process_folder(
    folder_path=folder_path,
    embedding_col=embedding_column,
    pattern='*.csv',
    chunksize=30000,
    output_format='csv',
    overwrite=True,
    in_place=True
)
folder_summary

Found 5 CSV files in: /home/hp/SEM2/INLP/Naretve_Shift/Processed_Data/Topic_Wise_w5
  [1] Climate.csv
  [2] Economics.csv
  [3] Health.csv
  [4] Technology.csv
  [5] War.csv

[1/5] Processing: Climate.csv
Processing file: Climate.csv
Done: Climate.csv | Saved: -35292037 bytes (-2.00%)

[2/5] Processing: Economics.csv
Processing file: Economics.csv
Done: Economics.csv | Saved: -16664569 bytes (-2.00%)

[3/5] Processing: Health.csv
Processing file: Health.csv
Done: Health.csv | Saved: -28540773 bytes (-2.01%)

[4/5] Processing: Technology.csv
Processing file: Technology.csv
Done: Technology.csv | Saved: -27119886 bytes (-2.00%)

[5/5] Processing: War.csv
Processing file: War.csv
Done: War.csv | Saved: -117309556 bytes (-2.00%)

=== Storage Improvement Summary ===
Total before: 11260793199 bytes
Total after : 11485720020 bytes
Total saved : -224926821 bytes (-2.00%)


,file,output,rows,parsed_ok,parsed_failed,size_before_bytes,size_after_bytes,saved_bytes,saved_pct,status
0,/home/hp/SEM2/INLP/Naretve_Shift/Processed_Dat...,/home/hp/SEM2/INLP/Naretve_Shift/Processed_Dat...,179681,179681,0,1766824451,1802116488,-35292037,-1.997484,success
1,/home/hp/SEM2/INLP/Naretve_Shift/Processed_Dat...,/home/hp/SEM2/INLP/Naretve_Shift/Processed_Dat...,84826,84826,0,833565490,850230059,-16664569,-1.999191,success
2,/home/hp/SEM2/INLP/Naretve_Shift/Processed_Dat...,/home/hp/SEM2/INLP/Naretve_Shift/Processed_Dat...,144841,144841,0,1423447051,1451987824,-28540773,-2.005046,success
3,/home/hp/SEM2/INLP/Naretve_Shift/Processed_Dat...,/home/hp/SEM2/INLP/Naretve_Shift/Processed_Dat...,138202,138202,0,1357096017,1384215903,-27119886,-1.998376,success
4,/home/hp/SEM2/INLP/Naretve_Shift/Processed_Dat...,/home/hp/SEM2/INLP/Naretve_Shift/Processed_Dat...,598527,598527,0,5879860190,5997169746,-117309556,-1.995108,success


,file,output,rows,parsed_ok,parsed_failed,size_before_bytes,size_after_bytes,saved_bytes,saved_pct,status
0,/home/hp/SEM2/INLP/Naretve_Shift/Processed_Dat...,/home/hp/SEM2/INLP/Naretve_Shift/Processed_Dat...,179681,179681,0,1766824451,1802116488,-35292037,-1.997484,success
1,/home/hp/SEM2/INLP/Naretve_Shift/Processed_Dat...,/home/hp/SEM2/INLP/Naretve_Shift/Processed_Dat...,84826,84826,0,833565490,850230059,-16664569,-1.999191,success
2,/home/hp/SEM2/INLP/Naretve_Shift/Processed_Dat...,/home/hp/SEM2/INLP/Naretve_Shift/Processed_Dat...,144841,144841,0,1423447051,1451987824,-28540773,-2.005046,success
3,/home/hp/SEM2/INLP/Naretve_Shift/Processed_Dat...,/home/hp/SEM2/INLP/Naretve_Shift/Processed_Dat...,138202,138202,0,1357096017,1384215903,-27119886,-1.998376,success
4,/home/hp/SEM2/INLP/Naretve_Shift/Processed_Dat...,/home/hp/SEM2/INLP/Naretve_Shift/Processed_Dat...,598527,598527,0,5879860190,5997169746,-117309556,-1.995108,success
